# 03 — Clustering

Cluster customers using the time-windowed behavioral features from notebook 02.

Steps:
1. Preprocess (log-transform skewed cols, StandardScaler)
2. K-Means sweep k=3..9 → pick best via silhouette + elbow
3. Agglomerative clustering as sanity check
4. PCA / UMAP visualisation
5. Extract **human-readable cluster profiles** → `outputs/cluster_profiles.json`
   (used directly as input to the LLM in notebook 04)

**Iteration rule:** If the LLM in notebook 04 finds clusters ambiguous,
come back here, increase `K_MIN` / `K_MAX`, re-run, then re-run notebook 04.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, pathlib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_palette('tab10')

features = pd.read_parquet('data/processed/customer_features.parquet')
CATEGORIES = ['entertainment','food_dining','gas_transport','grocery_net','grocery_pos',
              'health_fitness','home','kids_pets','misc_net','misc_pos',
              'personal_care','shopping_net','shopping_pos','travel']
WINDOWS = [6, 12]

print(f'Feature table: {features.shape[0]} customers × {features.shape[1]} features')

Feature table: 948 customers × 108 features


In [ ]:
import yaml

# ── Load pipeline configuration ───────────────────────────────────────────────
with open('config.yaml') as f:
    _cfg = yaml.safe_load(f)

N_CLUSTERS           = int(_cfg.get('n_clusters', 10))
CLUSTERING_ALGORITHM = str(_cfg.get('clustering_algorithm', 'hierarchical')).lower()

print(f'Config loaded:')
print(f'  n_clusters           = {N_CLUSTERS}')
print(f'  clustering_algorithm = {CLUSTERING_ALGORITHM}')

## 1. Preprocessing

In [2]:
# Log-transform right-skewed columns (counts and dollar amounts)
LOG_COLS = (
    [f'n_txn_{cat}_{w}m'     for cat in CATEGORIES for w in WINDOWS] +
    [f'amt_{cat}_{w}m'       for cat in CATEGORIES for w in WINDOWS] +
    [f'avg_spend_{cat}_{w}m' for cat in CATEGORIES for w in WINDOWS] +
    ['total_txn_count', 'total_spend', 'avg_txn_amt', 'std_txn_amt',
     'max_txn_amt', 'n_unique_merchants', 'avg_days_between_txn']
)
LOG_COLS = [c for c in LOG_COLS if c in features.columns]

X = features.copy()
for col in LOG_COLS:
    X[col] = np.log1p(X[col])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Scaled matrix: {X_scaled.shape}')

Scaled matrix: (948, 108)


## 2. Clustering

Fit the algorithm specified in `config.yaml` with the configured number of clusters.

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

if CLUSTERING_ALGORITHM == 'kmeans':
    _model = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=15, max_iter=500)
    cluster_labels = _model.fit_predict(X_scaled)
    algo_name   = 'KMeans'
    algo_detail = f'K-Means  |  k={N_CLUSTERS}  |  init=k-means++  |  n_init=15'

elif CLUSTERING_ALGORITHM == 'hierarchical':
    _model = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward')
    cluster_labels = _model.fit_predict(X_scaled)
    algo_name   = 'AgglomerativeClustering'
    algo_detail = f'Hierarchical (Ward linkage)  |  k={N_CLUSTERS}'

else:
    raise ValueError(
        f'Unknown clustering_algorithm: {CLUSTERING_ALGORITHM!r}. '
        'Valid options: "kmeans", "hierarchical".'
    )

sil = silhouette_score(X_scaled, cluster_labels)
palette = sns.color_palette('tab20', N_CLUSTERS)

features = features.copy()
features['cluster'] = cluster_labels

print(f'Algorithm   : {algo_detail}')
print(f'Silhouette  : {sil:.4f}')
print()
print('Cluster sizes:')
print(features['cluster'].value_counts().sort_index())

## 3. PCA Visualisation

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
for c in range(N_CLUSTERS):
    mask = cluster_labels == c
    plt.scatter(pca_coords[mask, 0], pca_coords[mask, 1],
                label=f'Cluster {c}', alpha=0.7, s=50, color=palette[c])
plt.title(f'PCA 2D — {N_CLUSTERS} Clusters ({CLUSTERING_ALGORITHM})')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('outputs/cluster_plots/pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
try:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    umap_coords = reducer.fit_transform(X_scaled)
    plt.figure(figsize=(10, 7))
    for c in range(N_CLUSTERS):
        mask = cluster_labels == c
        plt.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
                    label=f'Cluster {c}', alpha=0.7, s=50, color=palette[c])
    plt.title(f'UMAP 2D — {N_CLUSTERS} Clusters ({CLUSTERING_ALGORITHM})')
    plt.xlabel('UMAP-1'); plt.ylabel('UMAP-2')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.savefig('outputs/cluster_plots/umap_clusters.png', dpi=150, bbox_inches='tight')
    plt.show()
except ImportError:
    print('umap-learn not installed — skipping. Run: pip install umap-learn')

## 5. Cluster Profile Extraction

Build a **human-readable summary per cluster** that the LLM can interpret.
For each cluster we report, per category:
- Avg # transactions in past 12m and 6m
- Avg total spend in past 12m and 6m
- Avg spend per transaction (12m)
- Avg consecutive months (loyalty)

Plus overall metrics.

In [ ]:
profiles = {}

# Compute dataset-wide means for relative comparison
global_means = {
    cat: {
        'n_txn_12m'    : features[f'n_txn_{cat}_12m'].mean(),
        'total_amt_12m': features[f'amt_{cat}_12m'].mean(),
        'avg_spend_12m': features[f'avg_spend_{cat}_12m'].mean(),
        'consec_months': features[f'consec_months_{cat}'].mean(),
    }
    for cat in CATEGORIES
}

for c in range(N_CLUSTERS):
    grp = features[features['cluster'] == c]

    category_stats = {}
    for cat in CATEGORIES:
        n12   = grp[f'n_txn_{cat}_12m'].mean()
        a12   = grp[f'amt_{cat}_12m'].mean()
        avg12 = grp[f'avg_spend_{cat}_12m'].mean()
        n6    = grp[f'n_txn_{cat}_6m'].mean()
        a6    = grp[f'amt_{cat}_6m'].mean()
        cm    = grp[f'consec_months_{cat}'].mean()

        gm = global_means[cat]
        rel_n   = round(n12  / gm['n_txn_12m'],    2) if gm['n_txn_12m']    > 0 else 0
        rel_amt = round(a12  / gm['total_amt_12m'], 2) if gm['total_amt_12m'] > 0 else 0
        rel_cm  = round(cm   / gm['consec_months'], 2) if gm['consec_months'] > 0 else 0

        category_stats[cat] = {
            'n_txn_12m'    : round(n12,   1),
            'total_amt_12m': round(a12,   2),
            'avg_spend_12m': round(avg12, 2),
            'n_txn_6m'     : round(n6,    1),
            'total_amt_6m' : round(a6,    2),
            'consec_months': round(cm,    1),
            'rel_n_txn'    : rel_n,
            'rel_amt'      : rel_amt,
            'rel_consec'   : rel_cm,
        }

    # Sort by relative txn count — highlights what makes this cluster unique
    category_stats_sorted = dict(
        sorted(category_stats.items(), key=lambda x: -x[1]['rel_n_txn'])
    )

    profiles[str(c)] = {
        'cluster_id'         : c,
        'n_customers'        : len(grp),
        'clustering_algorithm': algo_name,
        'algorithm_detail'   : algo_detail,
        'category_stats'     : category_stats_sorted,
        'overall': {
            'avg_txn_amt'         : round(grp['avg_txn_amt'].mean(), 2),
            'std_txn_amt'         : round(grp['std_txn_amt'].mean(), 2),
            'max_txn_amt'         : round(grp['max_txn_amt'].mean(), 2),
            'pct_high_value'      : round(grp['pct_high_value'].mean() * 100, 1),
            'total_spend'         : round(grp['total_spend'].mean(), 2),
            'total_txn_count'     : round(grp['total_txn_count'].mean(), 1),
            'active_months'       : round(grp['active_months'].mean(), 1),
            'n_unique_categories' : round(grp['n_unique_categories'].mean(), 1),
            'n_unique_merchants'  : round(grp['n_unique_merchants'].mean(), 1),
            'avg_days_between_txn': round(grp['avg_days_between_txn'].mean(), 1),
        }
    }

pathlib.Path('outputs').mkdir(exist_ok=True)
with open('outputs/cluster_profiles.json', 'w') as f:
    json.dump(profiles, f, indent=2)

print(f'Saved outputs/cluster_profiles.json  ({N_CLUSTERS} clusters)')
print(f'Algorithm: {algo_detail}')
print('\nTop 3 relative differentiators per cluster:')
for cid, p in profiles.items():
    top3 = list(p['category_stats'].items())[:3]
    parts = [cat + '(' + str(s['rel_n_txn']) + 'x)' for cat, s in top3]
    print('  Cluster ' + cid + ': ' + ', '.join(parts))

In [ ]:
# Heatmap — mean 12m transaction count per cluster × category
hm_data = pd.DataFrame({
    cat: [profiles[str(c)]['category_stats'][cat]['n_txn_12m'] for c in range(N_CLUSTERS)]
    for cat in CATEGORIES
}, index=[f'Cluster {c}' for c in range(N_CLUSTERS)])

plt.figure(figsize=(16, max(4, N_CLUSTERS)))
sns.heatmap(hm_data, annot=True, fmt='.1f', cmap='YlOrRd',
            cbar_kws={'label': 'Avg # txns (12m)'})
plt.title(f'Mean # Transactions per Category (past 12m) by Cluster  [{CLUSTERING_ALGORITHM}]')
plt.tight_layout()
plt.savefig('outputs/cluster_plots/cluster_category_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save labelled features for classifier notebook
features.to_parquet('data/processed/customer_features_clustered.parquet')
print('Saved data/processed/customer_features_clustered.parquet')
print(f'Algorithm       : {algo_detail}')
print(f'Clusters        : {N_CLUSTERS}')
print(f'Silhouette score: {sil:.4f}')